# Experiments with boosting models

After modeling and error analysis, we explore additional experiments to improve predictions without significantly increasing computational cost. We focus on a boosting model that can capture numerical relationships CatBoost may miss, produce less-correlated errors, train efficiently, and handle high-dimensional feature spaces. 

These considerations lead us to LightGBM. We train LightGBM, compare its performance with CatBoost, and then evaluate whether combining their predictions can further improve our results.

## LightGBM and CatBoost 

This notebook compares LightGBM with the CatBoost model, then chooses a validation-based blend. It uses the same cleaned-enriched data and chronological split:

- training: January 2021 to March 2024
- validation: April 2024 to March 2025

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from time import perf_counter


TARGET = "MONTHLY_RENT"
train = pd.read_csv("derived/train_cleaned_enriched.csv")
test = pd.read_csv("derived/test_cleaned_enriched.csv")
print("Train shape:", train.shape)
print("Test shape: ", test.shape)

Train shape: (149841, 21)
Test shape:  (50000, 20)


In [2]:
# Match Models_main.ipynb exactly: hold out the latest 12 months.
approval_dates = pd.to_datetime(train["RENT_APPROVAL_DATE"])
cutoff = approval_dates.max().to_period("M").start_time - pd.DateOffset(months=12)
is_validation = approval_dates > cutoff
training_data = train.loc[~is_validation].copy()
validation_data = train.loc[is_validation].copy()
print(f"Cutoff month: {cutoff:%Y-%m}")
print(f"Training rows: {len(training_data):,}; validation rows: {len(validation_data):,}")

def split_xy(data):
    X = data.drop(columns=[TARGET], errors="ignore").copy()
    y = data[TARGET].copy() if TARGET in data.columns else None
    return X, y

X_fit_raw, y_fit = split_xy(training_data)
X_valid_raw, y_valid = split_xy(validation_data)
X_all_raw, y_all = split_xy(train)
X_test_raw, _ = split_xy(test)

Cutoff month: 2024-03
Training rows: 114,683; validation rows: 35,158


## Shared feature schema

CatBoost uses categorical columns natively. 

LightGBM receives a matrix made with the same numeric columns, one-hot encoded low-cardinality categories, and target-encoded `LOCATION_KEY`. Its preprocessor is fitted only on the training partition for validation, so validation targets cannot leak into features.

In [3]:
NUMERIC_FEATURES = [
    "FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "HDB_LATITUDE",
    "HDB_LONGITUDE", "HDB_YEAR_COMPLETED", "NEAREST_MRT_KM",
    "NEAREST_SCHOOL_KM", "NEAREST_MALL_KM", "APPROVAL_YEAR",
    "APPROVAL_MONTH", "MONTH_INDEX", "BUILDING_AGE",
]
ONEHOT_FEATURES = ["TOWN", "FLAT_TYPE", "FLAT_MODEL", "HDB_PLANNING_AREA"]
LOCATION_FEATURES = ["LOCATION_KEY"]
CATBOOST_ALL_FEATURES = NUMERIC_FEATURES + ONEHOT_FEATURES + LOCATION_FEATURES
CATBOOST_CATEGORICAL = ONEHOT_FEATURES + LOCATION_FEATURES

def make_lgbm_preprocessor():
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
    ])
    categorical = Pipeline([
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    location = Pipeline([
        ("encode", TargetEncoder(target_type="continuous", smooth="auto",
                                  cv=5, shuffle=True, random_state=42)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC_FEATURES),
        ("categorical", categorical, ONEHOT_FEATURES),
        ("location", location, LOCATION_FEATURES),
    ], remainder="drop", sparse_threshold=0)

preprocessor = make_lgbm_preprocessor()
X_fit_encoded = preprocessor.fit_transform(X_fit_raw, y_fit)
X_valid_encoded = preprocessor.transform(X_valid_raw)
print("Encoded training shape: ", X_fit_encoded.shape)
print("Encoded validation shape:", X_valid_encoded.shape)

Encoded training shape:  (114683, 95)
Encoded validation shape: (35158, 95)


/Users/apple/Documents/modules/CS5242/cs5242/lib/python3.14/site-packages/sklearn/preprocessing/_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


In [4]:
def make_catboost_features(data):
    X = data[CATBOOST_ALL_FEATURES].copy()
    for column in CATBOOST_CATEGORICAL:
        X[column] = X[column].fillna("<MISSING>").astype(str)
    for column in NUMERIC_FEATURES:
        X[column] = pd.to_numeric(X[column], errors="coerce")
        X[column] = X[column].fillna(X[column].median())
    return X

X_cb_fit = make_catboost_features(training_data)
X_cb_valid = make_catboost_features(validation_data)
X_cb_all = make_catboost_features(train)
X_cb_test = make_catboost_features(test)

# Same CatBoost settings as Models_main.ipynb.
# Best CatBoost parameters from Models_FLAML_tuning.ipynb (1094 selected trees).
CB_PARAMS = dict(loss_function="RMSE", eval_metric="RMSE", depth=7,
                  learning_rate=0.0718104404496359, l2_leaf_reg=10.104124680450155, random_seed=42,
                  thread_count=4, allow_writing_files=False)
catboost_validation_model = CatBoostRegressor(iterations=1094, **CB_PARAMS)

start = perf_counter()
catboost_validation_model.fit(
    X_cb_fit, y_fit, cat_features=CATBOOST_CATEGORICAL,
    eval_set=(X_cb_valid, y_valid), early_stopping_rounds=100,
    use_best_model=True, verbose=200)
cb_valid_pred = catboost_validation_model.predict(X_cb_valid)

catboost_time = perf_counter() - start
print(f"CatBoost training time: {catboost_time:.2f} seconds")

cb_iterations = catboost_validation_model.tree_count_
print("Selected CatBoost trees:", cb_iterations)


0:	learn: 705.5352214	test: 813.3009325	best: 813.3009325 (0)	total: 85.4ms	remaining: 1m 33s
200:	learn: 478.1289824	test: 476.9070470	best: 476.9069939 (199)	total: 3.91s	remaining: 17.4s
400:	learn: 473.4854045	test: 473.2782915	best: 473.2782915 (400)	total: 8.32s	remaining: 14.4s
600:	learn: 469.6777675	test: 471.9723036	best: 471.9707357 (595)	total: 13s	remaining: 10.7s
800:	learn: 466.3999820	test: 471.1947602	best: 471.1929223 (799)	total: 17.9s	remaining: 6.56s
1000:	learn: 463.7169411	test: 470.9405589	best: 470.9366214 (997)	total: 22.7s	remaining: 2.11s
1093:	learn: 462.4760268	test: 470.8198008	best: 470.8198008 (1093)	total: 25.1s	remaining: 0us

bestTest = 470.8198008
bestIteration = 1093

CatBoost training time: 25.51 seconds
Selected CatBoost trees: 1094


In [5]:
# Train directly with the saved best parameters; no tuning dependency is needed.
import json
from pathlib import Path

with Path("derived/flaml_tuning/best_parameters.json").open() as handle:
    LGBM_BEST_PARAMS = json.load(handle)["LightGBM"]
LGBM_FIXED = dict(objective="regression", random_state=42, n_jobs=4, verbosity=-1)

def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

# 449 is the tree count selected by early stopping during the completed search.
lightgbm_model = LGBMRegressor(**LGBM_FIXED, **LGBM_BEST_PARAMS)
start = perf_counter()
lightgbm_model.fit(X_fit_encoded, y_fit)
lgb_valid_pred = lightgbm_model.predict(X_valid_encoded)
lgbm_time = perf_counter() - start

display(pd.Series(LGBM_BEST_PARAMS, name="Selected value").to_frame())
print(f"LightGBM training time: {lgbm_time:.2f} seconds")
print("CatBoost validation RMSE: ", rmse(y_valid, cb_valid_pred))
print("LightGBM validation RMSE: ", rmse(y_valid, lgb_valid_pred))
print("LightGBM trees: ", LGBM_BEST_PARAMS["n_estimators"])


,Selected value
n_estimators,449.000000
learning_rate,0.069052
num_leaves,32.000000
min_child_samples,41.000000
subsample,0.800000
colsample_bytree,0.625312
reg_lambda,10.484056


LightGBM training time: 1.18 seconds
CatBoost validation RMSE:  470.8198008133436
LightGBM validation RMSE:  471.3617064821216
LightGBM trees:  449


LightGBM uses the saved best parameters, including 449 trees selected during the completed tuning run. Both the validation model and final full-data refit use these settings without rerunning hyperparameter search. The comparison below recomputes the blend with the selected CatBoost and LightGBM models. These remain model-selection scores because this validation set was used for tuning and blend-weight selection.

### Why a blend of both may work better

A blend can beat either model when their errors are not perfectly correlated. CatBoost and LightGBM both use boosted trees, but they construct useful partitions differently: CatBoost's ordered categorical statistics are useful for categories such as `LOCATION_KEY`, while LightGBM learns from a dense numeric/one-hot/target-encoded matrix and can make different split choices.

The weighted average can keep CatBoost's category-sensitive predictions while reducing some individual errors from either model. This is not guaranteed; the validation RMSE below is the evidence for this dataset. The weight is selected only from the shared time-based validation rows and then held fixed for the final test prediction.

In [6]:
best_blend = {"rmse": float("inf"), "catboost_weight": None, "prediction": None}
for cat_weight in np.linspace(0.0, 1.0, 21):
    blend_pred = cat_weight * cb_valid_pred + (1.0 - cat_weight) * lgb_valid_pred
    blend_rmse = rmse(y_valid, blend_pred)
    if blend_rmse < best_blend["rmse"]:
        best_blend = {"rmse": blend_rmse, "catboost_weight": float(cat_weight),
                      "prediction": blend_pred}

comparison = pd.DataFrame({
    "model": ["CatBoost", "LightGBM", "Blend"],
    "validation_RMSE": [rmse(y_valid, cb_valid_pred),
                          rmse(y_valid, lgb_valid_pred), best_blend["rmse"]]
})
display(comparison.sort_values("validation_RMSE").reset_index(drop=True))
print("Selected CatBoost weight:", best_blend["catboost_weight"])
print("Selected LightGBM weight: ", 1.0 - best_blend["catboost_weight"])

,model,validation_RMSE
0,Blend,469.583803
1,CatBoost,470.819801
2,LightGBM,471.361706


Selected CatBoost weight: 0.55
Selected LightGBM weight:  0.44999999999999996


In [7]:
# Refit both models on all labeled rows, using validation-selected tree counts and weight.
preprocessor_final = make_lgbm_preprocessor()
X_all_encoded = preprocessor_final.fit_transform(X_all_raw, y_all)
X_test_encoded = preprocessor_final.transform(X_test_raw)

final_catboost = CatBoostRegressor(iterations=cb_iterations, **CB_PARAMS)
final_catboost.fit(X_cb_all, y_all, cat_features=CATBOOST_CATEGORICAL, verbose=200)

final_lightgbm = LGBMRegressor(**LGBM_FIXED, **LGBM_BEST_PARAMS)
final_lightgbm.fit(X_all_encoded, y_all)

cat_weight = best_blend["catboost_weight"]
lgb_weight = 1.0 - cat_weight
cat_test_pred = final_catboost.predict(X_cb_test)
lgb_test_pred = final_lightgbm.predict(X_test_encoded)
final_test_pred = cat_weight * cat_test_pred + lgb_weight * lgb_test_pred

submission = pd.DataFrame({"Id": np.arange(len(final_test_pred)),
                           "Predicted": final_test_pred})
assert len(submission) == len(test)
assert submission["Predicted"].notna().all()
submission.to_csv("predictions/catboost_lightgbm_blend_submission.csv", index=False)
print("Saved predictions/catboost_lightgbm_blend_submission.csv")
display(submission.head())

/Users/apple/Documents/modules/CS5242/cs5242/lib/python3.14/site-packages/sklearn/preprocessing/_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


0:	learn: 722.0491399	total: 24.1ms	remaining: 26.3s
200:	learn: 477.0594540	total: 5.49s	remaining: 24.4s
400:	learn: 472.5019284	total: 11.7s	remaining: 20.2s
600:	learn: 468.7155174	total: 17.9s	remaining: 14.7s
800:	learn: 465.8442545	total: 24.1s	remaining: 8.81s
1000:	learn: 463.3406395	total: 30.6s	remaining: 2.84s
1093:	learn: 462.3102230	total: 33.6s	remaining: 0us
Saved predictions/catboost_lightgbm_blend_submission.csv


,Id,Predicted
0,0,2799.661830
1,1,3663.336084
2,2,3154.807307
3,3,3146.846821
4,4,3302.426224
